In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

# Preprocessing Pipeline Plan

- Limit observations to only the first encounter
- Drop additional prescribed medication due to limited clinical relevance
- Conduct feature decomposition on insulin and metformin to include a binary flag for being prescribed the medication and an ordinal encoding for the change
- Create train, validation and test splits
- Preprocess features
    - Convert data types (admission source, admission type)
    - Scale numeric features
    - Encode categoricals (OHE and ordinal)
    - Encode target (multi-class, binary)

## Experimenting with Feature Decomposition

Decomposing the insulin and metformin columns into two each: one binary flag to indicate if someone is on the drug at all, another ordinal encoding encoding if the patient is on the drug to indicate if there was a change in dosing.

In [ ]:
features = pd.read_parquet("data/diabetes_features.parquet")
target = pd.read_parquet("data/diabetes_target.parquet")

### Limiting the Data to the First Encounter Only

In [ ]:
# Creating a dataframe for selecting first encounters
splitting_df = features.copy()
splitting_df['target'] = (target['readmitted'] == "<30").astype(int)

# Keeping only the first encounter per patient_nbr
split_df = splitting_df.drop_duplicates('patient_nbr', keep='first')

### Preparing Features

Binary and Ordinal Splits for Insulin and Metformin

In [ ]:
# Creating a feature to indicate if a patient is not on insulin or metformin
split_df['no_insulin'] = (split_df.insulin == "No").astype(int)
split_df['no_metformin'] = (split_df.metformin == "No").astype(int)

# Creating a dictionary to encode changes in insulin and metformin
encode_dict = {'No': 0,
               "Steady": 0,
               "Up": 1,
               "Down": -1}

# Applying the encoding to engineer new features
split_df['metformin_change'] = split_df['metformin'].apply(lambda x: encode_dict[x])
split_df['insulin_change'] = split_df['insulin'].apply(lambda x: encode_dict[x])

Binary and Ordinal Splits for Other Columns with Missing Data

Columns to prepare: payer_code, medical_specialty

#### Creating Train and Test Splits

In [ ]:
# List of features to drop
drop_features = ['patient_nbr', 'encounter_id', 'target', 
                 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
                 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
                 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                 'tolazamide', 'examide', 'citoglipton', 'insulin',
                 'glyburide.metformin', 'glipizide.metformin',
                 'glimepiride.pioglitazone', 'metformin.rosiglitazone',
                 'metformin.pioglitazone', 'metformin', 'insulin']

# Building X & y dataframes
y = split_df['target']
X = split_df.drop(columns=drop_features)

# Creating train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

#### Setting up the Pipeline